# Qwen3-14B Jordan Peterson Fine-Tuning — Version 4
## Clean Q&A Data · Front + Back-Matter Removed · 3 Epochs · r=32 LoRA

This notebook fine-tunes Qwen3-14B on Jordan B. Peterson's four books using
Version 4 of the training pipeline.

**Pipeline position:** Run `JordanPeterson_DataPrep.ipynb` first to generate
the Q&A cache (`qa_dataset/peterson_qa.jsonl`).  This notebook reads from that
cache directly — no PDF extraction or question generation happens here.

**V4 = fine-tuning only.**  The dataset generation is handled by the standalone
DataPrep notebook.  V4 trains on a cleaner dataset (**3,936 pairs** from
**1,968 passages**) with both front-matter AND back-matter removed — eliminating
the index/endnote contamination observed in V3 inference.

---
## Version History

| Component | V1 | V2 | V3 | **V4** |
|-----------|----|----|----|--------|
| Training task | Passage completion | Synthetic Q&A | Synthetic Q&A | Synthetic Q&A |
| Dataset | Raw PDF (all pages) | Haiku Q&A (w/ front matter) | Haiku Q&A (front-matter removed) | **Haiku Q&A (front + back-matter removed)** |
| Passages | ~2,519 | ~2,519 | ~2,492 | **~1,968** |
| Q&A pairs | N/A | 5,029 | 4,867 | **3,936** |
| Epochs | 1 | 3 | 3 | **3** |
| LoRA rank | r=16 | r=32 | r=32 | **r=32** |
| LoRA alpha | 16 | 32 | 32 | **32** |
| Output adapter | `qwen3_14b_…_v1_lora` | `qwen3_14b_peterson_v2_lora` | `qwen3_14b_peterson_v3_lora` | **`qwen3_14b_peterson_v4_lora`** |

V4 uses identical hyperparameters to V3.  The only change is the training data:
the DataPrep notebook's new back-matter removal heuristic eliminates endnotes,
bibliography, references, and index sections from all four books before chunking.

---
## Why V4 Data Is Better Than V3's

### The Back-Matter Contamination Problem

The all-versions comparison (2026-02-21) showed that V3 had *worse* index
contamination than V2: **3 of 10 evaluation prompts** triggered raw index output
(e.g. `"143–74 and Genesis story, 160–68 in Harry Potter series, 259–60…"`)
versus just 1 in V2.

The cause: removing front matter without removing back matter **increased** the
proportional weight of index and endnote pages in the training corpus.  Less
front-matter → larger fraction of the remaining dataset is back-matter.

### What V4 Fixes

The DataPrep notebook now detects and excludes back-matter from each book:
endnotes, bibliography, references, and the full alphabetical index.

Detection strategy (last 30% of book only, to avoid mid-book false positives):
- **Signal 1**: ≥5 page-range patterns (e.g. `78–102`) AND numeric token
  density > 5%.  The density requirement prevents false positives from biblical
  verse ranges in *We Who Wrestle with God* (many `1-13` style references but
  near-zero standalone numeric tokens).
- **Signal 2**: numeric token density > 40% (unambiguously identifies index pages
  where the majority of tokens are bare page numbers).

**Validated back-matter cutoffs:**

| Book | Content pages | Back pages removed |
|------|--------------|-------------------|
| Maps of Meaning | 7–507 (82.5%) | 99 pages — Notes, References, Index |
| 12 Rules for Life | 19–372 (87.8%) | 30 pages — Endnotes, Index |
| Beyond Order | 10–316 (79.1%) | 71 pages — Notes, Index |
| We Who Wrestle with God | 8–564 (87.0%) | 75 pages — Notes, Index |

### Dataset Size Change

The removed back-matter pages were unusually word-dense: citation and index
pages average ~4× more words per page than prose pages.  This explains why
Maps of Meaning loses 779 pairs (1,821 → 1,042) despite "only" 16.5% of pages
being removed.

V4 has 931 fewer training pairs (3,936 vs 4,867) but all pairs are from book
body prose — none from indexes, endnotes, or reference lists.

In [ ]:
import json, re, math, time, gc
from pathlib import Path

import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# ── Load shared config ─────────────────────────────────────────────────────
# All paths and the system prompt are centralised in peterson_config.json so
# they stay in sync across the DataPrep and fine-tuning notebooks.
with open("peterson_config.json") as f:
    _config = json.load(f)

QA_CACHE      = Path(_config["paths"]["qa_cache"])
SYSTEM_PROMPT = _config["system_prompt"]

# ── V4 model + training constants ─────────────────────────────────────────
BASE_MODEL    = "unsloth/Qwen3-14B-unsloth-bnb-4bit"
OUTPUT_DIR    = Path("./outputs/qwen3_14b_peterson_v4_lora")   # does NOT touch V3
MAX_SEQ_LEN   = 2048
LORA_RANK     = 32
LORA_ALPHA    = 32
BATCH_SIZE    = 2
GRAD_ACCUM    = 4
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-4
WARMUP_STEPS  = 30
WEIGHT_DECAY  = 0.01

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}  |  GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print()
print("Configuration:")
print(f"  Base model  : {BASE_MODEL}")
print(f"  LoRA rank   : r={LORA_RANK}, alpha={LORA_ALPHA}  (ratio={LORA_ALPHA/LORA_RANK:.1f})")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  Batch (eff) : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR          : {LEARNING_RATE}")
print(f"  QA cache    : {QA_CACHE}")
print(f"  Output      : {OUTPUT_DIR.resolve()}")

---
# Part 1: Load the Q&A Dataset

The `JordanPeterson_DataPrep.ipynb` notebook generated this cache by:
1. Extracting ~350-word passages from each PDF with **front-matter and back-matter removal**
2. Calling Claude Haiku to generate 2 questions per passage

V3 trained on **4,867 pairs** from ~2,492 passages (front matter removed, back matter
still included).  V4 trains on **3,936 pairs** from ~1,968 passages — the reduction
comes entirely from removing endnotes, bibliography, and index sections that were
producing raw number output at inference time.

In [ ]:
from collections import Counter

if not QA_CACHE.exists():
    raise FileNotFoundError(
        f"Q&A cache not found at {QA_CACHE}. "
        "Run JordanPeterson_DataPrep.ipynb first to generate the dataset."
    )

with open(QA_CACHE) as f:
    records = [json.loads(line) for line in f if line.strip()]

# Build a HuggingFace Dataset from the flat list of dicts.
# We only keep "question" and "answer"; the "book" field is used for statistics.
raw_dataset = Dataset.from_list([
    {"question": r["question"], "answer": r["answer"]}
    for r in records
])

print(f"Dataset loaded: {len(raw_dataset):,} Q&A pairs")
print(f"Schema: {raw_dataset.column_names}")
print()

book_dist = Counter(r["book"] for r in records)
print("Distribution by book:")
for book, count in sorted(book_dist.items(), key=lambda x: -x[1]):
    print(f"  {book:<35}  {count:4d}  ({100*count/len(records):.1f}%)")

# V3 reference distribution for comparison
print()
print("V3 distribution (for comparison):")
v3_ref = {
    "Maps of Meaning": 1821,
    "We Who Wrestle with God": 1264,
    "12 Rules for Life": 926,
    "Beyond Order": 856,
}
for book, count in sorted(v3_ref.items(), key=lambda x: -x[1]):
    v4_count = book_dist.get(book, 0)
    delta = v4_count - count
    print(f"  {book:<35}  V3: {count:4d}  →  V4: {v4_count:4d}  ({delta:+d})")

---
# Part 2: Load the Qwen3-14B Base Model

## Why Qwen3-14B Over GPT-OSS 20B

Both models were evaluated in V1.  Qwen3-14B is the better choice for three
independent reasons:

1. **Lower training loss** (2.44 vs 3.01) — despite being a smaller model,
   Qwen3 absorbed the domain signal more efficiently in one epoch.

2. **Training speed** (23 min vs 73 min per epoch) — with 3 epochs planned,
   GPT-OSS would take ~220 min; Qwen3 takes ~70 min.

3. **Architectural recency** — Qwen3-14B (2025) is a more capable base model
   for instruction-following, which matters for the Q&A format we train on.

## Why `max_seq_length=2048`

Our training examples are ~350-word passages formatted as ChatML conversations.
After tokenisation, each example is typically 650–800 tokens.  Setting
`max_seq_length=2048` gives headroom for the system prompt and formatting
overhead without wasting VRAM on unnecessary KV-cache pre-allocation.

In [3]:
print(f"Loading {BASE_MODEL} ...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    dtype           = None,          # auto: bfloat16 on Ampere+, float16 otherwise
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,          # 4-bit quantization via bitsandbytes
    full_finetuning = False,         # we are adding LoRA, not full fine-tuning
)

vram_after_load = torch.cuda.memory_reserved() / 1e9
print(f"Model loaded.  VRAM reserved: {vram_after_load:.1f} GB")
print(f"Model dtype   : {next(model.parameters()).dtype}")
print(f"Tokenizer     : {tokenizer.__class__.__name__}")

Loading unsloth/Qwen3-14B-unsloth-bnb-4bit ...
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded.  VRAM reserved: 11.2 GB
Model dtype   : torch.bfloat16
Tokenizer     : Qwen2TokenizerFast


---
# Part 3: Add LoRA Adapters (r=32)

## What LoRA Does

Low-Rank Adaptation freezes the pre-trained weight matrices `W` and injects a
trainable update `ΔW = (alpha/r) × B × A` where:
- `B` has shape `(d_model × r)` — initialised to zero
- `A` has shape `(r × d_model)` — initialised with random Gaussian values
- `r` is the rank

At the start of training `B=0` so `ΔW=0` and the model behaves identically
to the base model.  Training gradually fills `B` with the directions needed
to capture Peterson's style.

## Why r=32 vs V1's r=16

Think of `r` as the number of "stylistic dimensions" the adapter can learn.
With r=16, the adapter has 16 orthogonal directions to encode things like:
*"use chaos/order vocabulary"*, *"frame arguments historically"*,
*"connect psychology to mythology"*, etc.

With r=32 we have 32 such dimensions — enough to capture both the macro-level
thematic vocabulary AND the micro-level sentence rhythm that makes Peterson's
writing recognisable.  The adapter checkpoint grows from ~260 MB to ~520 MB;
VRAM impact is negligible.

## Target Modules

We apply LoRA to all attention projections (q, k, v, o) and all MLP
gate/up/down projections.  This is the full set recommended by Unsloth for
stylistic fine-tuning because style is encoded across both attention patterns
(what the model attends to) and MLP activations (how it transforms those
representations).

Setting `alpha = rank` (alpha=32, rank=32) keeps the effective scale factor
at `alpha/rank = 1.0` — the LoRA update is applied at the same relative
strength as the base weights, which is the established baseline.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_RANK,    # 32 — doubled from V1's r=16
    lora_alpha          = LORA_ALPHA,   # 32 — alpha/rank = 1.0 (same scale as V1)
    lora_dropout        = 0,            # 0 is optimal for Unsloth according to docs
    target_modules      = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # all attention projections
        "gate_proj", "up_proj", "down_proj",        # all MLP projections
    ],
    use_gradient_checkpointing = "unsloth",   # saves ~30% VRAM vs PyTorch default
    random_state        = 42,
    use_rslora          = False,   # standard LoRA (not rank-stabilised variant)
    loftq_config        = None,    # no quantization-aware init needed with 4-bit base
)

# ── Count trainable parameters ─────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct_trainable    = 100 * trainable_params / total_params

print(f"Total parameters    : {total_params:>15,}")
print(f"Trainable (LoRA)    : {trainable_params:>15,}  ({pct_trainable:.4f}% of total)")
print(f"  V1 had r=16: ~{trainable_params//2:,} trainable params")
print(f"  V3 has r=32: ~{trainable_params:,} trainable params (2x capacity)")
print(f"VRAM after LoRA     : {torch.cuda.memory_reserved()/1e9:.1f} GB")

Unsloth 2026.2.1 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


Total parameters    :   8,691,809,280
Trainable (LoRA)    :     128,450,560  (1.4778% of total)
  V1 had r=16: ~64,225,280 trainable params
  V3 has r=32: ~128,450,560 trainable params (2x capacity)
VRAM after LoRA     : 11.7 GB


---
# Part 4: Format Dataset into Qwen3 ChatML Format

Each Q&A pair is converted into a three-turn conversation using the Qwen3
ChatML template.  The format change from V1 to V2/V3:

| Role | V1 content | **V2/V3 content** |
|------|-----------|----------------|
| `system` | Generic assistant prompt | **Peterson persona prompt** |
| `user` | A book passage chunk | **A synthetically generated question** |
| `assistant` | Continuation of the passage | **The original passage as an answer** |

The system prompt changes from generic to persona-specific so the model
learns: *"when given the Peterson persona instruction, respond in his voice"*.

The user message changes from a passage fragment to a real question so the
model learns: *"when asked a question in this domain, produce a substantive
Peterson-style answer"*.

## About `enable_thinking=False`

We format training examples with `enable_thinking=False`.  This tells Qwen3's
chat template not to include chain-of-thought reasoning tokens in the formatted
string.  Even with this setting, the template adds an empty
`<think>\n\n</think>` block before the assistant response — this is expected
and will be masked out by `train_on_responses_only` in the next step, so it
does not contribute to the loss.

## System Prompt Source

`SYSTEM_PROMPT` is loaded from `peterson_config.json` (not hardcoded) so
the DataPrep notebook and all fine-tuning notebooks use identical persona
instructions.

In [5]:
def format_example(batch):
    '''
    Convert a batch of Q&A records into formatted Qwen3 ChatML strings.

    Each formatted string looks like:
        <|im_start|>system
        You are an AI assistant... (Peterson persona)
        <|im_end|>
        <|im_start|>user
        <the generated question>
        <|im_end|>
        <|im_start|>assistant
        <think>

        </think>
        <the Peterson passage>
        <|im_end|>

    The <think></think> block appears because Qwen3's template always includes
    a thinking placeholder even when enable_thinking=False.  It is masked out
    during training by train_on_responses_only and does not affect the loss.
    '''
    formatted_texts = []
    for question, answer in zip(batch["question"], batch["answer"]):
        conversation = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer},
        ]
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize              = False,
            add_generation_prompt = False,
            enable_thinking       = False,
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}


dataset = raw_dataset.map(
    format_example,
    batched=True,
    batch_size=256,
    remove_columns=raw_dataset.column_names,
)

print(f"Formatted dataset: {len(dataset):,} examples")
print(f"Columns: {dataset.column_names}")
print()
print("-" * 70)
print("SAMPLE FORMATTED TRAINING EXAMPLE")
print("-" * 70)
sample = dataset[0]["text"]
print(sample[:800])
print("...")
print("-" * 70)
print(f"Full sample length: {len(sample.split())} words / {len(sample)} chars")

Map:   0%|          | 0/4867 [00:00<?, ? examples/s]

Formatted dataset: 4,867 examples
Columns: ['text']

----------------------------------------------------------------------
SAMPLE FORMATTED TRAINING EXAMPLE
----------------------------------------------------------------------
<|im_start|>system
You are an AI assistant that has been trained on the complete works of Jordan B. Peterson, a Canadian clinical psychologist, professor, and author. You speak with deep knowledge of psychology, philosophy, mythology, religion, and personal responsibility. Your responses reflect Peterson's writing style, intellectual depth, and interdisciplinary approach to understanding human nature and meaning.<|im_end|>
<|im_start|>user
How does the brain's neurological structure relate to our psychological capacity to generate meaning from chaos?<|im_end|>
<|im_start|>assistant
<think>

</think>

FIGURES 1 The Domain and Constituent Elements of the Known 15 2 The Metamythological Cycle of the Way 17 3 Normal Life 28 4 Revolutionary Adaptation 31 5 The Ambiv

In [6]:
# Auto-detect the response boundary tokens by inspecting the formatted data.
# This guards against future Qwen3 template changes that might shift the token
# boundaries, and mirrors the robust approach used in V2.
_sample_text = dataset[0]["text"]

if "<|im_start|>assistant\n" in _sample_text:
    instruction_part = "<|im_start|>user\n"
    response_part    = "<|im_start|>assistant\n"
    print("Detected Qwen3 ChatML response boundary tokens:")
else:
    raise RuntimeError(
        "Could not detect Qwen3 ChatML tokens in formatted dataset. "
        "Check the output of format_example() above."
    )

print(f"  instruction_part : {repr(instruction_part)}")
print(f"  response_part    : {repr(response_part)}")
print()
print("Boundary tokens set — train_on_responses_only will be applied below.")

Detected Qwen3 ChatML response boundary tokens:
  instruction_part : '<|im_start|>user\n'
  response_part    : '<|im_start|>assistant\n'

Boundary tokens set — train_on_responses_only will be applied below.


---
# Part 5: Configure and Run Training

## SFTConfig Parameters Explained

- **`num_train_epochs=3`**: The most important change from V1.  Three full
  passes over the dataset let the model move from memorisation into genuine
  stylistic generalisation.  The empirical sweet spot for LoRA stylistic
  fine-tuning: 1 epoch = memorisation, 3 epochs = generalisation, 5+ = risk
  of overfitting.

- **`warmup_steps=30`**: Gradually ramps the learning rate from 0 → 2e-4
  over the first 30 gradient steps (~3% of total training).  Prevents the
  randomly-initialised LoRA matrices from making destructively large weight
  updates at the start.

- **`lr_scheduler_type="cosine"`**: After warmup the learning rate decays
  smoothly following a cosine curve, reaching ~0 at the end of epoch 3.
  Cosine decay is generally more stable than linear decay for LoRA.

- **`weight_decay=0.01`**: Mild L2 regularisation on the LoRA weights.
  Discourages the adapter from memorising individual passages and encourages
  generalisation.

- **`optim="adamw_8bit"`**: Unsloth's 8-bit AdamW keeps optimizer states in
  8-bit rather than 32-bit, saving ~3 GB of VRAM with no measurable quality
  cost for LoRA training.

- **`packing=False`**: Sequence packing combines multiple short examples into
  one long sequence for efficiency.  We disable it because our examples are
  long (~400 words → ~700 tokens), and packing them could leak context between
  unrelated Q&A pairs.

## V4 Expected Step Count

With 3,936 pairs (vs V3's 4,867), the expected gradient updates are:

```
ceil(3936 / 2) × 3 // 4 = 1968 × 3 // 4 = 5904 // 4 = 1,476
```

V3 ran **1,827 steps** from 4,867 pairs — V4 should run approximately
**1,476 steps**, a reduction reflecting the smaller (but cleaner) dataset.

In [7]:
sft_trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    args             = SFTConfig(
        dataset_text_field          = "text",
        max_length                  = MAX_SEQ_LEN,
        dataset_num_proc            = 2,

        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs            = NUM_EPOCHS,

        learning_rate               = LEARNING_RATE,
        warmup_steps                = WARMUP_STEPS,
        lr_scheduler_type           = "cosine",
        weight_decay                = WEIGHT_DECAY,

        optim                       = "adamw_8bit",
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),

        logging_steps               = 25,
        save_strategy               = "epoch",
        output_dir                  = str(OUTPUT_DIR),
        report_to                   = "none",

        seed                        = 42,
        packing                     = False,
    ),
)

sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part = instruction_part,
    response_part    = response_part,
)

# ── Masking verification ───────────────────────────────────────────────────
# If masking is working, labels will be -100 for all system/user tokens and
# real token IDs only for the assistant response tokens.
_sample_input = sft_trainer.train_dataset[0]
_n_total      = len(_sample_input["input_ids"])
_n_trained    = sum(1 for lbl in _sample_input["labels"] if lbl != -100)
_n_masked     = _n_total - _n_trained
print("Response masking check on first example:")
print(f"  Total tokens   : {_n_total}")
print(f"  Trained tokens : {_n_trained}  (assistant response only)")
print(f"  Masked tokens  : {_n_masked}   (system + user = {100*_n_masked/_n_total:.0f}% of input)")
print()

_total_steps = math.ceil(len(dataset) / BATCH_SIZE) * NUM_EPOCHS // GRAD_ACCUM
print(f"Estimated gradient updates: {_total_steps:,}")
print(f"  ({len(dataset):,} examples / {BATCH_SIZE} batch x {GRAD_ACCUM} accum x {NUM_EPOCHS} epochs)")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4867 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=20):   0%|          | 0/4867 [00:00<?, ? examples/s]

Filter (num_proc=20):   0%|          | 0/4867 [00:00<?, ? examples/s]

Response masking check on first example:
  Total tokens   : 744
  Trained tokens : 645  (assistant response only)
  Masked tokens  : 99   (system + user = 13% of input)

Estimated gradient updates: 1,825
  (4,867 examples / 2 batch x 4 accum x 3 epochs)


In [ ]:
import time

vram_before = torch.cuda.memory_reserved() / 1e9
print(f"VRAM before training: {vram_before:.1f} GB")
print(f"Starting training — {NUM_EPOCHS} epochs, ~{_total_steps:,} gradient updates...")
print()

t0 = time.time()
train_result = sft_trainer.train()
elapsed_min  = (time.time() - t0) / 60

vram_peak = torch.cuda.max_memory_reserved() / 1e9

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Steps          : {train_result.global_step:,}")
print(f"  Training loss  : {train_result.training_loss:.4f}")
print(f"  Elapsed        : {elapsed_min:.1f} min")
print(f"  Peak VRAM      : {vram_peak:.1f} GB")
print()
print("V3 reference:")
print("  Steps: 1,827  |  Loss: 1.5258  |  Time: 136.9 min  |  VRAM: 15.3 GB")
print()
print("V2 reference:")
print("  Steps: 1,887  |  Loss: 1.5058  |  Time: 144.0 min  |  VRAM: 15.5 GB")



---
# Part 6: Save the LoRA Adapter

We save only the LoRA adapter weights — not the full 14B model.  This keeps
the checkpoint small (~520 MB for r=32 vs ~28 GB for a full model copy).
At inference time Unsloth loads the base model from HuggingFace Hub and
merges the adapter on-the-fly.

**V4 output path**: `outputs/qwen3_14b_peterson_v4_lora/`

The V3 adapter at `outputs/qwen3_14b_peterson_v3_lora/` is **not touched**.
All adapters coexist and can be compared in the all-versions comparison notebook.

In [ ]:
print(f"Saving V4 LoRA adapter to {OUTPUT_DIR} ...")

model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

# List saved files with sizes
adapter_files = (
    list(OUTPUT_DIR.glob("*.safetensors"))
    + list(OUTPUT_DIR.glob("*.bin"))
)
total_mb = sum(f.stat().st_size for f in adapter_files) / 1e6

print()
print("Adapter files:")
for f in sorted(adapter_files):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

print()
print(f"Total adapter size: {total_mb:.1f} MB")
print(f"  (V3 r=32 was 513.9 MB; V4 r=32 should be identical)")
print()
print(f"V3 adapter (untouched) : outputs/qwen3_14b_peterson_v3_lora/")
print(f"V4 adapter (just saved): {OUTPUT_DIR}")

---
# Part 7: Inference Test

The same 5 evaluation prompts are used here as in V1, V2, and V3 so results are
directly comparable.  Copy-paste the outputs into a comparison doc alongside
V3 outputs to spot-check whether the front-matter fix produces visibly better
responses (especially for Maps of Meaning questions).

We run greedy decoding (`do_sample=False`) for deterministic, reproducible
outputs — the same setting used in all comparison notebooks.

In [ ]:
FastLanguageModel.for_inference(model)

EVAL_PROMPTS = [
    "What is the relationship between order and chaos in human experience?",
    "Why is personal responsibility the foundation of a meaningful life?",
    "How do ancient myths and stories reveal truths about human nature?",
    "What does it mean to pursue what is meaningful rather than what is expedient?",
    "How should a person confront suffering rather than flee from it?",
]


def ask_v4(question: str, max_new_tokens: int = 300) -> str:
    '''
    Generate a response from the V3 fine-tuned model.

    Uses greedy decoding (do_sample=False) for reproducibility, consistent
    with how all prior version outputs are evaluated in the comparison notebooks.
    enable_thinking=False keeps reasoning mode disabled, matching training.
    '''
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = False,
            temperature        = 1.0,
            repetition_penalty = 1.1,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Strip any residual thinking blocks
    response   = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    return response


print("Testing V4 fine-tuned model (greedy decoding)...")
print()
for i, prompt in enumerate(EVAL_PROMPTS):
    print("-" * 70)
    print(f"Q{i+1}: {prompt}")
    print("-" * 70)
    answer = ask_v4(prompt)
    print(answer if answer.strip() else "(empty response)")
    print()

---
# Conclusions

## What V4 Changed vs V3

V4 uses identical hyperparameters to V3 (r=32, 3 epochs, 2e-4 LR).  The only
change is the training data: the DataPrep notebook's new back-matter removal
heuristic eliminates endnotes, bibliography, references, and alphabetical index
sections from all four books before chunking.

The net effect: **931 fewer Q&A pairs** (4,867 → 3,936), all subtracted from
citation-heavy and index-heavy back-matter.  No philosophical content was removed.

## Expected Improvement

The primary expected improvement is the **elimination of index-page output**.
V3's 3/10 index-contaminated prompts (Q2, Q6, Q7 producing raw number sequences)
should drop to near zero in V4 — the training corpus no longer contains any raw
index entries, endnote numbers, or reference lists.

The slightly lower step count (~1,476 vs V3's 1,827) may reduce the risk of
overfitting, partly offsetting the smaller dataset size.

**Residual issue**: the Maps of Meaning figure list (page 7, just after front
matter ends) is still the first chunk and contributes ~3 contaminated pairs out
of 3,936 — a minor remaining artefact.

## Adding V4 to the All-Versions Comparison Notebook

To compare V4 against base, V1, V2, and V3:

1. Open `Qwen3_14B_AllVersions_JordanPeterson_Comparison.ipynb`
2. Add `"v4"` to `MODEL_KEYS` (after `"v3"`)
3. Add `"v4": "V4 (back-matter removed)"` to `MODEL_LABELS`
4. Add `"v4": "./outputs/qwen3_14b_peterson_v4_lora/"` to `MODEL_PATHS`
5. Add a colour entry for `"v4"` in `MODEL_COLORS`
6. Delete `comparison_cache_qwen3_versions/v4_results.pkl` if it exists
7. Re-run the comparison notebook

## Next Levers If Further Improvement Is Needed

| Option | Expected gain | Effort |
|--------|--------------|--------|
| Fix Maps figure-list (page 7) | Remove ~3 contaminated pairs | Low |
| 5 epochs (instead of 3) | Marginal style improvement; risk of overfitting | Low |
| 3 questions per passage | More data diversity (~6,000 pairs) | Medium |
| `r=64, alpha=64` | More adapter capacity; +200 MB checkpoint | Low |
| Human-edited answers | Highest quality ceiling | High |

In [ ]:
!nvidia-smi